### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [70]:
import polars as pl
import pyarrow
import pandas as pd

In [71]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [72]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [73]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [68]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
MetaData=(
    pl.scan_parquet(folder / 'MetaData.parquet')
        .filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))
)

In [69]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')

CPU times: user 35 μs, sys: 50 μs, total: 85 μs
Wall time: 88.2 μs
